# 07 - Treinamento do classificador

**Objetivo:** Comparar árvore de decisão e Random Forest e avaliar as classes por janela.

> Execute as células na ordem. As saídas são gravadas nas pastas do projeto.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Localiza a raiz do projeto independentemente de o notebook ser aberto pela pasta raiz ou notebooks.
CANDIDATOS=[Path.cwd(), Path.cwd().parent]
ROOT=next((p.resolve() for p in CANDIDATOS if (p/'dados').exists() and (p/'notebooks').exists()),None)
if ROOT is None: raise FileNotFoundError('Abra o VS Code na pasta projeto_soft_sensor.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.funcoes import *
sns.set_theme(style='whitegrid')
print('Raiz:',ROOT)


## Carregamento

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay,f1_score
import joblib
b=pd.read_csv(ROOT/'dados/03_processados/base_classificacao_janelas.csv',encoding='utf-8-sig'); b=b[b.janela_amostras.eq(12)].copy(); feats=[c for c in ['media','mediana','minimo','maximo','desvio_padrao','variancia','amplitude','iqr','dif_media_abs','dif_max_abs','var_diferencas','prop_dif_zero','valores_unicos','inclinacao','dif_media_externa','correlacao_externa']]
tr=b.particao.eq('treino'); va=b.particao.eq('validacao'); te=b.particao.eq('teste')

## Treinamento e seleção

In [ ]:
models={'Arvore':Pipeline([('imp',SimpleImputer(strategy='median')),('m',DecisionTreeClassifier(max_depth=6,class_weight='balanced',random_state=42))]),'Random Forest':Pipeline([('imp',SimpleImputer(strategy='median')),('m',RandomForestClassifier(n_estimators=240,max_depth=12,min_samples_leaf=2,class_weight='balanced',n_jobs=-1,random_state=42))])}
rows=[]
for name,m in models.items():
    m.fit(b.loc[tr,feats],b.loc[tr,'classe']); p=m.predict(b.loc[va,feats]); rows.append({'modelo':name,'F1_macro':f1_score(b.loc[va,'classe'],p,average='macro')})
sel=pd.DataFrame(rows).sort_values('F1_macro',ascending=False); display(sel); best_name=sel.iloc[0].modelo; best=models[best_name]

## Teste final

In [ ]:
pred=best.predict(b.loc[te,feats]); rel=classification_report(b.loc[te,'classe'],pred,labels=[0,1,2],target_names=['Normal','Travado','Ruido'],output_dict=True,zero_division=0); display(pd.DataFrame(rel).T)
pd.DataFrame(rel).T.to_csv(ROOT/'resultados/metricas/metricas_classificador.csv',encoding='utf-8-sig')
cm=confusion_matrix(b.loc[te,'classe'],pred,labels=[0,1,2]); ConfusionMatrixDisplay(cm,display_labels=['Normal','Travado','Ruido']).plot(cmap='Blues'); plt.tight_layout(); plt.savefig(ROOT/'resultados/figuras/classificacao/matriz_confusao.png',dpi=150); plt.show()

## Importância e salvamento

In [ ]:
model=best.named_steps['m']; imp=pd.Series(model.feature_importances_,index=feats).sort_values(ascending=False); display(imp.to_frame('importancia')); imp.to_csv(ROOT/'resultados/tabelas/importancia_features_classificador.csv',header=['importancia'])
plt.figure(figsize=(8,5)); imp.sort_values().plot.barh(); plt.tight_layout(); plt.savefig(ROOT/'resultados/figuras/classificacao/importancia_features.png',dpi=150); plt.show()
joblib.dump({'modelo':best,'features':feats,'nome':best_name,'janela':12},ROOT/'resultados/modelos/classificador_falhas.joblib'); print('Modelo salvo:',best_name)

## Conclusões da etapa

Execute todas as células, registre as observações exibidas e confirme os arquivos gerados antes de avançar para o notebook seguinte.